In [1]:
import sqlite3
import requests
import time

# -----------------------
# Danh sách quốc gia
# -----------------------
country_list = [
    ("NZ", "New Zealand"),
    ("AU", "Australia"),
    ("US", "United States"),
    ("CA", "Canada"),
    ("GB", "United Kingdom"),
    ("DE", "Germany"),
    ("FR", "France"),
    ("IT", "Italy"),
    ("ES", "Spain"),
    ("JP", "Japan"),
    ("KR", "South Korea"),
    ("CN", "China"),
    ("IN", "India"),
    ("SG", "Singapore")
]

# -----------------------
# Kết nối DB
# -----------------------
DB_FILE = "universities_db.db"
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

In [2]:
# -----------------------
# Tạo bảng countries
# -----------------------
cur.execute("""
CREATE TABLE IF NOT EXISTS countries (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    code TEXT UNIQUE NOT NULL,
    name TEXT NOT NULL,
    flag_url TEXT
)
""")

# -----------------------
# Tạo bảng universities
# -----------------------
cur.execute("""
CREATE TABLE IF NOT EXISTS universities (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    country_id INTEGER,
    state TEXT,
    domain TEXT,
    website TEXT,
    num_majors INTEGER,
    tuition_fee_avg REAL,
    entry_requirements TEXT,
    FOREIGN KEY(country_id) REFERENCES countries(id)
)
""")
conn.commit()

# -----------------------
# Thêm quốc gia vào DB
# -----------------------
for code, name in country_list:
    cur.execute(
        "INSERT OR IGNORE INTO countries (code, name, flag_url) VALUES (?, ?, ?)",
        (code.lower(), name, f"https://flags.com/{code.lower()}.png")
    )
conn.commit()



In [3]:
# -----------------------
# Hàm lấy JSON an toàn
# -----------------------
def safe_get_json(url, retries=3):
    for attempt in range(retries):
        try:
            res = requests.get(url, timeout=10)
            if res.status_code != 200:
                print("HTTP error:", res.status_code)
                continue
            if "application/json" not in res.headers.get("Content-Type", ""):
                print("Not JSON:", res.headers.get("Content-Type"))
                continue
            return res.json()
        except Exception as e:
            print(f"JSON parse failed (attempt {attempt+1}):", e)
            time.sleep(1)
    return []  # return empty list if fail



In [4]:
# -----------------------
# Fetch data và chèn vào DB
# -----------------------
for code, name in country_list:
    print(f"Fetching: {name} ...")
    url = f"http://universities.hipolabs.com/search?country={name.replace(' ', '+')}"
    data = safe_get_json(url)

    if not data:
        print(f"!! Failed to fetch JSON for {name}, skipping...")
        continue

    # Lấy id quốc gia
    cur.execute("SELECT id FROM countries WHERE code = ?", (code.lower(),))
    country_id_row = cur.fetchone()
    if not country_id_row:
        print(f"!! Country {name} not found in DB, skipping...")
        continue
    country_id = country_id_row[0]

    for uni in data:
        state = uni.get("state-province") or None
        domain = (uni.get("domains") or [None])[0]
        website = (uni.get("web_pages") or [None])[0]

        cur.execute("""
            INSERT INTO universities (
                name, country_id, state, domain, website,
                num_majors, tuition_fee_avg, entry_requirements
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            uni.get("name"),
            country_id,
            state,
            domain,
            website,
            None,  # num_majors
            None,  # tuition_fee_avg
            None   # entry_requirements
        ))

    conn.commit()

conn.close()
print("DONE")

Fetching: New Zealand ...
Fetching: Australia ...
Fetching: United States ...
JSON parse failed (attempt 1): ('Connection broken: IncompleteRead(142106 bytes read, 295680 more expected)', IncompleteRead(142106 bytes read, 295680 more expected))
JSON parse failed (attempt 2): ('Connection broken: IncompleteRead(142106 bytes read, 295680 more expected)', IncompleteRead(142106 bytes read, 295680 more expected))
JSON parse failed (attempt 3): ('Connection broken: IncompleteRead(142106 bytes read, 295680 more expected)', IncompleteRead(142106 bytes read, 295680 more expected))
!! Failed to fetch JSON for United States, skipping...
Fetching: Canada ...
Fetching: United Kingdom ...
Fetching: Germany ...
Fetching: France ...
Fetching: Italy ...
Fetching: Spain ...
Fetching: Japan ...
Fetching: South Korea ...
!! Failed to fetch JSON for South Korea, skipping...
Fetching: China ...
Fetching: India ...
Fetching: Singapore ...
DONE
